In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import OneClassSVM
from sklearn import svm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split, KFold, GridSearchCV


#HUSK AT ÆNDRE PATH HER!!
df=pd.read_csv(r"C:\Users\Esben\OneDrive\8. semester\Computional data analysis\project_2\HR_data.csv",index_col=0) 

In [9]:
features=['HR_TD_Mean','HR_TD_std','TEMP_TD_Mean','TEMP_TD_std','EDA_TD_P_Mean','EDA_TD_T_Mean']
df_clean=df.dropna(subset=features+['Phase', 'Frustrated']).copy()
print("clean")

clean


In [20]:
train=df_clean[df_clean['Phase']=='phase1'][features]
scaler=StandardScaler()
X_train=scaler.fit_transform(train)
model=OneClassSVM(kernel='rbf')
model.fit(X_train)
print("trained")

trained


vi skal lige vælge rigtig kernal og måske andet, RBF bliver brugt i slides


In [ ]:
#Use this function to evaluate how good model is. used for tuning hyperparameters
#should probably find a better way to score.
#We should find another method for scoring 'how good' our model is a detecting anamolies
def score_model(model):
    p2=df_clean[df_clean['Phase']=='phase2'].copy()
    p2['pred']=model.predict(scaler.transform(p2[features]))
    flagged=(p2['pred']==-1).sum()
    print(f"phase2 anomaly %:{flagged}/{len(p2)} ({100*flagged/len(p2):.1f}%)")
    return flagged/len(p2)

In [ ]:
## here we run custom cross-validation fold to find best hyper parameters for model
best_model = None
best_score = 0.0

# Initialize KFold cross-validator
kf = KFold(n_splits=5, shuffle=True, random_state=42) 


best_params = []

kernelType = ["rbf", "poly", "sigmoid"]
                
nuVal = np.linspace(0.01, 0.9, 10)  # Generate 10 values between 0.01 and 1.0

for kernel in kernelType:
    for nu in nuVal:
        fold_scores = []
        # Perform cross-validation
        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]


        # Train the model on the training fold
            model = OneClassSVM(kernel=kernel, degree=3, nu=nu)
            model.fit(X_train_fold)
            
            # Evaluate the model on the validation fold
            val_score = score_model(model)  # this evaluation is not good
            fold_scores.append(val_score)
        
        # Average the scores across folds
        avg_score = np.mean(fold_scores)
        
        if avg_score > best_score: #this doesnt make sense
            best_score = avg_score
            best_model = model
            best_params = model.get_params()

print(f"Best Model: {best_model}, Best Score: {best_score}")

phase2 anomaly %:14/104 (13.5%)
phase2 anomaly %:27/104 (26.0%)
phase2 anomaly %:27/104 (26.0%)
phase2 anomaly %:21/104 (20.2%)
phase2 anomaly %:22/104 (21.2%)
phase2 anomaly %:14/104 (13.5%)
phase2 anomaly %:27/104 (26.0%)
phase2 anomaly %:27/104 (26.0%)
phase2 anomaly %:20/104 (19.2%)
phase2 anomaly %:22/104 (21.2%)
phase2 anomaly %:18/104 (17.3%)
phase2 anomaly %:27/104 (26.0%)
phase2 anomaly %:28/104 (26.9%)
phase2 anomaly %:24/104 (23.1%)
phase2 anomaly %:28/104 (26.9%)
phase2 anomaly %:29/104 (27.9%)
phase2 anomaly %:36/104 (34.6%)
phase2 anomaly %:37/104 (35.6%)
phase2 anomaly %:38/104 (36.5%)
phase2 anomaly %:34/104 (32.7%)
phase2 anomaly %:39/104 (37.5%)
phase2 anomaly %:43/104 (41.3%)
phase2 anomaly %:42/104 (40.4%)
phase2 anomaly %:42/104 (40.4%)
phase2 anomaly %:41/104 (39.4%)
phase2 anomaly %:46/104 (44.2%)
phase2 anomaly %:48/104 (46.2%)
phase2 anomaly %:45/104 (43.3%)
phase2 anomaly %:48/104 (46.2%)
phase2 anomaly %:45/104 (43.3%)
phase2 anomaly %:53/104 (51.0%)
phase2 a